# DFX4ML Multi-Region System Test

End-to-end test for a **multi-PR-region** DFX4ML system.

The system uses **partial reconfiguration** to dynamically swap hardware accelerator kernels
(Reconfigurable Modules, RMs) across multiple independent PR regions at runtime.
The test pipeline:

1. Load the static full bitstream (base overlay)
2. Initialise the DFX Manager (sequencer) and DFX Controller
3. Load partial bitstreams for each region into CMA memory
4. Feed input data through the pipeline via DMA
5. The DFX Manager orchestrates: load → compute → store → reconfigure → repeat
6. Read back results and verify correctness

> **Hardware Setup Required:** This notebook must run on a PYNQ board with the exported bitstreams in `hw/`.

## Step 1 — Project Configuration

In [ ]:
import os
import time
import asyncio
import numpy as np
from pynq import Overlay, allocate, Interrupt

# do not remove this import
import driver.cap           as cap
import driver.mem_alloc     as dataAlloc
import driver.dfx_unified   as dfx_unified
import driver.dfx_mgs_debug as dfx_mgs_debug

# ── paths ─────────────────────────────────────────────────────────────────
PRJ_DIR    = os.getcwd()
PRJ_HW_DIR = os.path.join(PRJ_DIR, 'hw')
PRJ_TC_DIR = os.path.join(PRJ_DIR, 'data')

# ── system configuration ──────────────────────────────────────────────────
# Must match the values used when building (num_pr_region / rm_index_width).
NUM_PR_REGION = 2          # number of independent reconfigurable regions
AMT_SLOT      = 2          # number of session slots in the DFX Manager

# ── bitstream names ───────────────────────────────────────────────────────
FULL_BS_NAME = 'system.bin'

# Partial bitstream for region r, RM m  →  hw/region_{r}_rm_{m}.bin
# Here we load RM 0 for every region.
PAR_BS = {r: os.path.join(PRJ_HW_DIR, f'region_{r}_rm_0.bin') for r in range(NUM_PR_REGION)}

# ── data parameters ───────────────────────────────────────────────────────
INPUT_DATA_NAME = 'input_x.npy'
AMT_QUERY       = 100
INPUT_SHAPE     = (AMT_QUERY, 1)
OUTPUT_SHAPE    = (AMT_QUERY, 1)

# ── physical address map (must match board_build.tcl / HWH) ──────────────
DMA_PHY_ADDR     = 0xA003_0000
PR_CTRL_PHY_BASE = 0xA005_0000   # region 0; region r → base + r * stride
PR_CTRL_STRIDE   = 0x0001_0000

pr_phy_addrs = [PR_CTRL_PHY_BASE + r * PR_CTRL_STRIDE for r in range(NUM_PR_REGION)]
print('PR ctrl physical addresses:', [hex(a) for a in pr_phy_addrs])

In [ ]:
# Generate synthetic input data and save to disk
input_x = (np.arange(AMT_QUERY, dtype=np.int32) + 48).reshape(INPUT_SHAPE)
np.save(os.path.join(PRJ_TC_DIR, INPUT_DATA_NAME), input_x)

## Step 2 — Load the Full Bitstream (Static Overlay)

In [ ]:
cap.change_pl_config_mode('pcap', True, '')
overlay = Overlay(os.path.join(PRJ_HW_DIR, FULL_BS_NAME))
print('Overlay loaded.')

## Step 3 — Register Interrupt

In [ ]:
overlay.interrupt_pins
my_interrupt = Interrupt('dfx_unified_0/dfx_intr')

## Step 4 — Access IP Sub-blocks

| Handle | Role |
|---|---|
| `dfx_mng`  | DFX Manager — orchestrates DMA → compute → reconfig sequence |
| `dfx_ctrl` | DFX Controller — drives ICAP for partial reconfiguration |
| `dfx_dma`  | AXI DMA — moves data between DDR and PR regions |
| `dfx_man`  | PR Decoupler / Reset manager (single instance, all regions) |
| `pr_ctrl[r]` | Per-region HLS kernel control (AP_CTRL interface) |

In [ ]:
dfx_ip  = overlay.dfx_unified_0

dfx_mng  = dfx_ip.dfx_mng
dfx_ctrl = dfx_ip.dfx_ctrl
dfx_dma  = dfx_ip.dfx_dma
dfx_man  = dfx_ip.dfx_man

# Per-region pr_ctrl — raises RuntimeError if region is still decoupled.
# Access only after releasing the decoupler for that region.
print(f'NUM_PR_REGION = {dfx_ip.NUM_PR_REGION}')
print(f'LIM_AMT_SLOT  = {1 << dfx_ip.SLOT_INDEX_WIDTH}')

## Step 5 — Configure DFX Controller

In [ ]:
DFX_CONFIG_FILE = 'dfx_ctrl_con.txt'
dfx_ctrl.config(os.path.join(PRJ_HW_DIR, DFX_CONFIG_FILE))
print('dfx_ctrl configured. BLS_REGID =', dfx_ctrl.BLS_REGID)

# Switch PL config interface to ICAP for runtime partial reconfiguration.
cap.change_pl_config_mode('icap', True, '')

## Step 6 — Initial System Reset

In [ ]:
dfx_mng.shutdown_engine()
for r in range(NUM_PR_REGION):
    dfx_ctrl.shutdown_engine(r)
    dfx_ctrl.print_status(r)

## Step 7 — Initialise DFX Manager (Bank 0 Metadata)

Configure the sequencer with data-source addresses, session count,
and the physical address of the DMA and PR-ctrl IPs.

In [ ]:
print('------ before init ------')
dfx_mng.print_debug()

print('------ init Bank 0 metadata ------')
dfx_mng.set_last_session(2)   # last valid session index
dfx_mng.set_dma_ip_addr(DMA_PHY_ADDR)
dfx_mng.set_pr_ip_addr(pr_phy_addrs[0])  # region-0 pr_ctrl; HW uses stride for others
dfx_mng.set_amt_query(AMT_QUERY)
dfx_mng.set_amt_query_per_iter(AMT_QUERY)
dfx_mng.set_intr_ena(1)

In [ ]:
# Allocate input / output CMA buffers
inputX = np.load(os.path.join(PRJ_TC_DIR, INPUT_DATA_NAME))
assert inputX.shape == INPUT_SHAPE, f'Shape mismatch: {inputX.shape} vs {INPUT_SHAPE}'

buf_input, buf_input_phya, buf_input_sz = dataAlloc.alloc_data_uint(
    alloc_shape=INPUT_SHAPE,  alloc_type=np.int32, input_x=inputX)
buf_out,   buf_out_phy,   buf_out_sz   = dataAlloc.alloc_data_uint(
    alloc_shape=OUTPUT_SHAPE, alloc_type=np.int32)
buf_input.flush()
print('CMA buffers allocated.')
print(f'  input  @ {hex(buf_input_phya)}, size={hex(buf_input_sz)}')
print(f'  output @ {hex(buf_out_phy)},   size={hex(buf_out_sz)}')

In [ ]:
# Configure slot table (Bank 1)
# Field order: [src_addr, src_size, des_addr, des_size,
#               prof_recon, prof_exec, vs_rm_recon_sel, vs_rm_exec_sel,
#               load_mask, store_mask, complete_mask, next_session]

# Slot 0: recon only
dfx_mng.set_whole_slot(0, [
    0   ,          0, # src (none)
    0   ,          0, # dst (none)
    0   ,          0, # profile counters
    0b01,       0b00, # RM select (recon) (exec) (RM 0)
    0   ,          0, # load mask (none), store mask (ch0)
    0   ,             # complete mask
    1   ,             # next_session → slot 0 (wrap)
])

# Slot 1: DMA-load input, execute with RM 0, no store yet → advance to slot 1
dfx_mng.set_whole_slot(1, [
    buf_input_phya, buf_input_sz,   # src
    0             ,         0   ,   # dst (none)
    0             ,         0   ,   # profile counters
    0b10          ,      0b01   ,   # RM select for recon / exec (RM 0)
    0b01          ,      0b10   ,   # load mask (ch0), store mask (none)
    0             ,                 # complete mask
    2             ,                 # next_session → slot 1
])

# Slot 2: no load, execute with RM 0, DMA-store output → wrap back to slot 0
dfx_mng.set_whole_slot(2, [
    0          ,              0, # src (none)
    buf_out_phy,     buf_out_sz, # dst
    0          ,              0, # profile counters
    0b00       ,           0b10, # RM select (RM 0)
    0b10       ,           0b01, # load mask (none), store mask (ch0)
    0          ,                 # complete mask
    0          ,                 # next_session → slot 0 (wrap)
])

print('------ after slot init ------')
dfx_mng.print_debug()

## Step 8 — Inspect PR Region State

Grant PS control of the decoupler so we can safely access each region's pr_ctrl.

In [ ]:
dfx_man.grant_decoupler_to_ps()

for r in range(NUM_PR_REGION):
    dfx_man.release_decup(region=r)   # make region accessible
    print(f'--- PR Region {r} ---')
    dfx_ip.get_pr_ctrl(r).print_status()

## Step 9 — Load Partial Bitstreams into CMA

Allocate CMA buffers for each region's RM bitstream and register them with the DFX Controller.

In [ ]:
par_bs_bufs = {}   # region → (buf, addr, size)

for r in range(NUM_PR_REGION):
    buf, addr, size = dfx_ctrl.allocate_bit_stream_cma(PAR_BS[r])
    par_bs_bufs[r] = (buf, addr, size)
    print(f'  region {r}: {PAR_BS[r]}  @ {hex(addr)}, size={hex(size)}')

In [ ]:
# Register each bitstream with the DFX Controller.
# slot_id = region index; each region starts with bitstream index 0 (its only RM).
for r in range(NUM_PR_REGION):
    _, addr, size = par_bs_bufs[r]
    dfx_ctrl.set_simple_meta_data(r, 0, addr, size)   # (slot_id, bs_idx, addr, size)

for r in range(NUM_PR_REGION):
    dfx_ctrl.print_status(r)
    dfx_ctrl.print_simple_meta_data(r, 0)

## Step 10 — Initial PR Trigger (Load RM into Each Region)

Hand decoupler control to the DFX Controller, then trigger reconfiguration
for each region so every region has a valid bitstream before we start.

In [ ]:
for r in range(NUM_PR_REGION):
    dfx_man.hold_decup(region=r)

In [ ]:
for r in range(NUM_PR_REGION):
    print(f'Loading RM into region {r} ...')
    dfx_ctrl.trig(r, 0)             # slot_id=r, trigger_id=0
    dfx_ctrl.restart_no_status(r)   # slot_id=r

for r in range(NUM_PR_REGION):
    dfx_ctrl.print_status(r)

In [ ]:
dfx_man.grant_decoupler_to_dfx_ctrl()

## Step 11 — Execute Pipeline and Wait for Interrupt

In [ ]:
async def run_and_wait():
    start = time.perf_counter()
    dfx_mng.start_engine()
    await my_interrupt.wait()
    elapsed = time.perf_counter() - start
    print(f'Interrupt received. Elapsed: {elapsed:.6f} s')

loop = asyncio.get_event_loop()
loop.run_until_complete(run_and_wait())

In [ ]:
dfx_mng.print_debug()
dfx_mng.shutdown_engine()
dfx_mng.print_debug()

## Step 12 — Read Back and Verify Results

In [ ]:
buf_out.invalidate()
np_result = np.array(buf_out, dtype=np.int32)
print(np_result)
print(f'\noutput == input_x: {np.array_equal(input_x, np_result)}')

## Step 13 — Per-Region Status (After Run)

Re-grant PS control so we can inspect each region's HLS AP-ctrl registers.

In [ ]:
dfx_man.grant_decoupler_to_ps()

for r in range(NUM_PR_REGION):
    dfx_man.release_decup(region=r)
    print(f'--- PR Region {r} ---')
    dfx_ip.get_pr_ctrl(r).print_status()

In [ ]:
dbg_val = overlay.mgs_debugger.read(0)

In [ ]:
load_amount  = (dbg_val >>  0) & 0x7FF
store_amount = (dbg_val >> 11) & 0x7FF
state        = (dbg_val >> 22) & 0x1F
print(f'dbg_val      = {hex(dbg_val)}')
print(f'load_amount  = {load_amount}')
print(f'store_amount = {store_amount}')
print(f'state        = {state} (0x{state:02X})')

In [ ]:
print("mm2s_status")
print(dfx_dma.check_mm2s_status())

In [ ]:
print(dfx_dma.read(dfx_dma.MM2S_LENGTH))

In [ ]:
print(hex(dfx_dma.read(dfx_dma.MM2S_SA)))

In [ ]:
print("s2mm_status")
print(dfx_dma.check_s2mm_status())

In [ ]:
print(dfx_dma.read(dfx_dma.S2MM_LENGTH))

In [ ]:
print(hex(dfx_dma.read(dfx_dma.S2MM_DA)))